In [0]:
#standardization and replacement
#standardization1 - Column Enrichment (Addition of columns)
#create new column with default value
from pyspark.sql.functions import col,lit,upper

standdf1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/staging/custs",inferSchema=True).toDF("custid","fname","lname","age","profession")
standdf2=standdf1.withColumn("source",lit("raw"))
#display(standdf2)

#Standardization2 - Column Uniformity
#profession column Uniformity with upper
standdf3=standdf2.withColumn("profession",upper("profession"))
#display(standdf3.limit(20))

#Standardization3 - Format Standardization
#check id and age column if it contains non integer values
standdf3.where("custid rlike '[a-zA-Z]'").show()
standdf3.where("age rlike '[^0-9]'").show()

standdf3.printSchema()

#replace ten with 10 using replace function 
#regreplace of age by removing - between 4-7
from pyspark.sql.functions import replace,regexp_replace
replacedict={'one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9','ten':'10'}
standdf4=standdf3.na.replace(replacedict,["custid"])
#standdf4.where("custid rlike '[a-zA-Z]'").show()
standdf4.where("custid='10'").show()

standdf5=standdf4.withColumn('age',regexp_replace(col('age'),"-",""))
#standdf5.where("age rlike '[^0-9]'").show()
standdf5.where("age=47").show(10,False)

In [0]:
#Standardization4 - Data Type Standardization
standdf5.printSchema()

standdf6=standdf5.withColumn("custid",standdf5['custid'].cast('long'))
standdf6=standdf6.withColumn("age",standdf5['age'].cast('short'))
standdf6.printSchema()

In [0]:
#Standardization5 - Naming Standardization
#rename the existing columns

standdf7=standdf6.withColumnsRenamed({'custid':'CustomerID','fname':'FirstName','lname':'LastName','age':'Age','profession':'Profession','source':'Source'})
display(standdf7.limit(10))


In [0]:
#Standardization6 - Reorder Standadization
#original column order in dataframe
#display(standdf7).limit(10)
#reorder
#display(standdf7.select("Age","Profession","FirstName","LastName","CustomerID","Source").limit(10))

#drop column using na.drop
standdf8=standdf7.na.drop(subset=['profession']) #this will remove only the null value present in the profession column
display(standdf8.count())

standdf9=standdf8.drop("Source")
display(standdf9)


In [0]:
#Passive data munging
#high level
#schema1="custid string,fname string,lname string,age string,profession string"
rawdf1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB1 Customers/",inferSchema=True, pathGlobFilter="cust*",recursiveFileLookup=True).toDF("custid","fname","lname","age","profession")
#rawdf1.show(10,False) > Pyspark show dont Uses driver memory and not interactive and Limited formatting. take file data into python list of values in a dataframe.
#display(rawdf1.take(10)) > Databricks take Uses driver memory but more interactive, display function render them in a table and more formatting
'''
rawdf1.printSchema()
print(rawdf1.columns)
print(rawdf1.dtypes)
print(rawdf1.schema)

#count
display(rawdf1.count())

#summary and description
display(rawdf1.describe())
display(rawdf1.summary())
'''
#remove duplicates
from pyspark.sql.functions import col

display(rawdf1.count()) #gives total number of rows in dataframe includes duplicates
display(rawdf1.groupBy("custid").count().orderBy(col("custid").asc()))

#display(rawdf1.distinct().count()) #gives total number of rows in dataframe after removing duplicates entire row
#display(rawdf1.dropDuplicates().count()) #gives total number of rows in dataframe after removing duplicates in a column level
display(rawdf1.dropDuplicates(['custid']).count()) #removes duplicate on specified column 'custid'




In [0]:
#Active data munging
#Take data from two difference source and merge/melt it togather
from pyspark.sql.types import StructType,StructField,StringType
nyschema=StructType([StructField('custid', StringType(), True), StructField('fname', StringType(), True), StructField('lname', StringType(), True), StructField('age', StringType(), True), StructField('profession', StringType(), True)])

nydf1=spark.read.schema(nyschema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",pathGlobFilter="*_NY",recursiveFileLookup=True)
display(nydf1)

txschema="custid int,fname string, age int,profession string,lname string"
txdf2=spark.read.schema(txschema).format("csv").options(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",pathGlobFilter="*_TX",recursiveFileLookup=True).load()
display(txdf2)

ny_txdf=nydf1.unionByName(txdf2,allowMissingColumns=True)
display(ny_txdf)

#implement schema evalution
#step-1: read file as a dataframe
txschema1="custid int,fname string, age int,profession string,lname string,city string"
txdf3=spark.read.schema(txschema1).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/custsmodified_TX1.txt")
display(txdf3)

#step-2: write the above two dataframe's into orc/parquet format
txdf2.write.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mode="append")
txdf3.write.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mode="append")

#step-3: read the orc data with mergeschema=True to get newly added columns from the source
finaltxdf=spark.read.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mergeSchema=True)
display(finaltxdf)

In [0]:
#Rejection Strategy
newschema="id string,fname string,lname string,age string,profession string,corruptedcol string"
df01=spark.read.schema(newschema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB1 Customers/custsmodified",mode="permissive",columnNameOfCorruptRecord="corruptedcol")

print("Original Dataframe Count:",df01.count())
#method-1 using where function
data_without_correupted_col=df01.where("corruptedcol IS NOT NULL")
display(data_without_correupted_col)

#method-2: using drop function
df02=data_without_correupted_col.na.drop(subset=["corruptedcol"])
display(df02)

#multiple columns with null
df03=df02.na.drop(how='any',subset=["age","profession"])
display(df03)

newdf04=df01.na.drop(subset=["profession"])
display(newdf04.count())

#remove duplicate of ID column
newdf05=df01.dropDuplicates(['id'])
display(newdf05.count())